## 1. Dataset Loading and Initial Cleaning

In [1]:
import polars as pl

# Load the raw cleaned Ethereum transactions dataset
df = pl.read_csv("eth_tx_last4days_clean.csv")
df.head()

# Process and normalize transaction events
events = (
    df
    .select([
        pl.col("hash"),
        pl.col("from_address").str.to_lowercase().alias("source"),  # Normalize addresses to lowercase
        pl.col("to_address").str.to_lowercase().alias("target"),
        pl.col("value").alias("value_wei"),
        pl.col("block_timestamp")
    ])
    .with_columns([
        pl.col("target").fill_null("__contract_creation__"),        # Handle null values from contract deployments
        (
            pl.col("block_timestamp")
            .str.to_datetime(format="%Y-%m-%d %H:%M:%S%z")
            .dt.timestamp("ms") // 1000                             # Convert block time to UNIX epoch seconds
        ).alias("timestamp"),
        (
            pl.col("value_wei").cast(pl.Float64, strict=False) / 1_000_000_000_000_000_000
        ).alias("value_eth")                                         # Convert Wei to Ether values
    ])
    .select([
        "hash", "source", "target", "timestamp", "block_timestamp", "value_wei", "value_eth"
    ])
    .sort("timestamp")                                               # Ensure sequential temporal processing
)

events.head()

hash,source,target,timestamp,block_timestamp,value_wei,value_eth
str,str,str,i64,str,f64,f64
"""0x0c2085c64ffd00db90fed2023ab7…","""0xe264ee25f98ef65c11e9f285ee74…","""0x6131b5fae19ea4f9d964eac0408e…",1762546907,"""2025-11-07 20:21:47+00:00""",0.0,0.0
"""0xbfcc1aa0065f83cce790ec1dba18…","""0x2ebe9c09bfff2c9782cf9560fe5a…","""0xdac17f958d2ee523a22062069945…",1762546907,"""2025-11-07 20:21:47+00:00""",0.0,0.0
"""0x65fe12b67c23b8edb4184307cd10…","""0xbe84d31b2ee049dcb1d8e7c79851…","""0x000000fee13a103a10d593b9ae06…",1762546907,"""2025-11-07 20:21:47+00:00""",0.0,0.0
"""0x7f713adbb337d5dcd504e2a6e4f1…","""0xd6295b77cd029a45310a130ae440…","""0x1ab4973a48dc892cd9971ece8e01…",1762546907,"""2025-11-07 20:21:47+00:00""",3.1532e17,0.315323
"""0x14312968be330fe1f67c723ebd1f…","""0x8afaacdfaab4a6938a2da1705a6a…","""0x1275727a3ac42add3ed38c4f690c…",1762546907,"""2025-11-07 20:21:47+00:00""",8.0600e13,0.000081


## 2. Data Validation and Descriptive Statistics

In [2]:
# Display dataset shape and inferred schema types
print(events.shape)
print(events.schema)

# Verify there are no remaining missing values in required features
events.select([
    pl.col("timestamp").null_count().alias("missing_timestamp"),
    pl.col("source").null_count().alias("missing_source"),
    pl.col("target").null_count().alias("missing_target"),
    pl.col("value_wei").null_count().alias("missing_value_wei"),
    pl.col("value_eth").null_count().alias("missing_value_eth")
])


(4290480, 7)
Schema([('hash', String), ('source', String), ('target', String), ('timestamp', Int64), ('block_timestamp', String), ('value_wei', Float64), ('value_eth', Float64)])


missing_timestamp,missing_source,missing_target,missing_value_wei,missing_value_eth
u32,u32,u32,u32,u32
0,0,0,0,0


In [3]:
# Profile the dataset range (min/max bounds and basic distribution stats)
events.select([
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
    pl.col("block_timestamp").first().alias("first_block_timestamp"),
    pl.col("block_timestamp").last().alias("last_block_timestamp"),
    pl.col("value_eth").min().alias("min_value_eth"),
    pl.col("value_eth").max().alias("max_value_eth"),
    pl.col("value_eth").mean().alias("mean_value_eth")
])

min_timestamp,max_timestamp,first_block_timestamp,last_block_timestamp,min_value_eth,max_value_eth,mean_value_eth
i64,i64,str,str,f64,f64,f64
1762546907,1762793675,"""2025-11-07 20:21:47+00:00""","""2025-11-10 16:54:35+00:00""",0.0,39448.75,0.688368


## 3. Extracting Unique Network Nodes

In [4]:
print("Collecting active addresses from transactions...")

# Extract all unique interactive endpoints (addresses) present in the network
nodes = (
    pl.concat([
        events.select(pl.col("source").alias("address")),
        events.select(pl.col("target").alias("address"))
    ])
    .unique()
)

print(f"Base 'nodes' table created. Total active nodes: {nodes.height}")

Base 'nodes' table created. Total active nodes: 1397958


## 4. Repeated Same-Direction Motif Features 

In [5]:
DELTA_SECONDS = 3600  # Define temporal proximity threshold (1 hour)

# Detect sequences where sender and receiver repeat an active transaction within the window
repeated_same_events = (
    events
    .sort(["source", "target", "timestamp"])
    .with_columns([
        pl.col("timestamp").shift(1).over(["source", "target"]).alias("prev_ts"),   # Fetch preceding transaction timestamp
        pl.col("value_eth").shift(1).over(["source", "target"]).alias("prev_val")
    ])
    .with_columns(
        (pl.col("timestamp") - pl.col("prev_ts") <= DELTA_SECONDS).alias("is_motif") # Verify time constraint
    )
    .filter(pl.col("is_motif") == True)
)

# Aggregate repeated events from the perspective of the sender
repeated_sender_features = (
    repeated_same_events
    .group_by("source")
    .agg([
        pl.len().alias("repeat_same_direction_as_sender_count_1h"),
        pl.col("value_eth").sum().alias("repeat_same_direction_as_sender_value_eth_1h")
    ])
    .rename({"source": "address"})
)

# Aggregate repeated events from the perspective of the receiver
repeated_receiver_features = (
    repeated_same_events
    .group_by("target")
    .agg([
        pl.len().alias("repeat_same_direction_as_receiver_count_1h"),
        pl.col("value_eth").sum().alias("repeat_same_direction_as_receiver_value_eth_1h")
    ])
    .rename({"target": "address"})
)

# Map features back to structural node base
nodes = (
    nodes
    .join(repeated_sender_features, on="address", how="left")
    .join(repeated_receiver_features, on="address", how="left")
    .fill_null(0) # Maintain mathematical completeness
)

nodes.shape

(1397958, 5)

## 5. Reciprocity Motif Features

In [6]:
# Group bi-directional communication channels safely regardless of orientation order
reciprocity_data = (
    events
    .with_columns([
        pl.when(pl.col("source") < pl.col("target"))
        .then(pl.col("source")).otherwise(pl.col("target")).alias("p1"),
        pl.when(pl.col("source") < pl.col("target"))
        .then(pl.col("target")).otherwise(pl.col("source")).alias("p2")
    ])
    .sort(["p1", "p2", "timestamp"])
    .with_columns([
        pl.col("source").shift(1).over(["p1", "p2"]).alias("prev_source"),          # Determine the last message orientation
        pl.col("timestamp").shift(1).over(["p1", "p2"]).alias("prev_ts"),
        pl.col("value_eth").shift(1).over(["p1", "p2"]).alias("prev_val")
    ])
    .with_columns(
        ((pl.col("source") != pl.col("prev_source")) &                               # Verify interaction is a response from the alternate party
         (pl.col("timestamp") - pl.col("prev_ts") <= DELTA_SECONDS))
        .alias("is_reciprocity")
    )
    .filter(pl.col("is_reciprocity") == True)
)

# Accumulate frequency and value distribution profiles for reciprocal actions
recip_features = (
    reciprocity_data
    .group_by("source")
    .agg([
        pl.len().alias("motif_reciprocity_cnt"),
        pl.col("value_eth").sum().alias("motif_reciprocity_val_sum")
    ])
    .rename({"source": "address"})
)

# Merge structural updates into nodes frame
nodes = (
    nodes
    .join(recip_features, on="address", how="left")
    .fill_null(0)
)

nodes.shape

(1397958, 7)

## 6. Chain and Structural Fan-In / Fan-Out Motifs

In [7]:
from collections import defaultdict
from bisect import bisect_left, bisect_right

# Count chain motifs where the current address acts as the middle node: A -> address -> B
def count_chain_middle_safe(group, delta=3600):
    address = group["address"][0]
    
    # Split transactions into incoming and outgoing events for this address
    incoming = group.filter(pl.col("direction") == "incoming").sort("timestamp")
    outgoing = group.filter(pl.col("direction") == "outgoing").sort("timestamp")
    
    # If one direction is missing, no chain motif can be formed
    if incoming.height == 0 or outgoing.height == 0:
        return pl.DataFrame({"address": [address], "motif_chain_cnt": [0]})

    in_times = incoming["timestamp"].to_list()
    out_times = outgoing["timestamp"].to_list()
    
    # For each incoming transaction, count outgoing transactions within the time window
    chain_count = 0
    for t_in in in_times:
        left = bisect_right(out_times, t_in)
        right = bisect_right(out_times, t_in + delta)
        chain_count += (right - left)
        
    return pl.DataFrame({"address": [address], "motif_chain_cnt": [chain_count]})



# Create an address-level event table with transaction direction
chain_input = pl.concat([
    events.select([
        pl.col("target").alias("address"),
        pl.lit("incoming").alias("direction"),
        pl.col("timestamp"),
        pl.col("value_eth")
    ]),
    events.select([
        pl.col("source").alias("address"),
        pl.lit("outgoing").alias("direction"),
        pl.col("timestamp"),
        pl.col("value_eth")
    ])
])

# Compute chain motif count for each address independently
chain_features = (
    chain_input
    .group_by("address")
    .map_groups(lambda g: count_chain_middle_safe(g, delta=3600))
)



# Count incoming star-like behavior: many sources sending to the same address
fan_in_features = (
    events.group_by("target")
    .agg([
        pl.col("source").n_unique().alias("motif_fan_in_unique_sources"),
        pl.len().alias("motif_fan_in_cnt")
    ])
    .rename({"target": "address"})
)

# Count outgoing star-like behavior: one address sending to many targets
fan_out_features = (
    events.group_by("source")
    .agg([
        pl.col("target").n_unique().alias("motif_fan_out_unique_targets"),
        pl.len().alias("motif_fan_out_cnt")
    ])
    .rename({"source": "address"})
)

# Add motif-based features to the node table
nodes = (
    nodes
    .join(chain_features, on="address", how="left")
    .join(fan_in_features, on="address", how="left")
    .join(fan_out_features, on="address", how="left")
    .fill_null(0)
)

nodes.shape

(1397958, 12)

## 7. Temporal Cycle Motif Features

In [8]:
import polars as pl
from collections import defaultdict
from bisect import bisect_left, bisect_right

# Return an empty DataFrame with the expected schema if no cycles are found
def empty_cycle_contribs():
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "cycle_as_start_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_start_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "cycle_as_middle_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_middle_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "cycle_as_end_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_end_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })

# Build lookup indexes to search previous and closing edges efficiently
def build_cycle_indexes(tx):
    incoming_by_target = defaultdict(lambda: {"times": [], "sources": [], "values": []})
    closing_by_pair = defaultdict(lambda: {"times": [], "values": []})

    tx_sorted = tx.sort("timestamp")

    # Store incoming transactions for each target and all transactions for each ordered pair
    for source, target, timestamp, value_eth in tx_sorted.iter_rows():
        incoming_by_target[target]["times"].append(timestamp)
        incoming_by_target[target]["sources"].append(source)
        incoming_by_target[target]["values"].append(value_eth)

        closing_by_pair[(source, target)]["times"].append(timestamp)
        closing_by_pair[(source, target)]["values"].append(value_eth)

    # Build prefix sums of values for fast value aggregation over time intervals
    closing_prefix_by_pair = {}
    for pair, data in closing_by_pair.items():
        prefix = [0.0]
        for value in data["values"]:
            prefix.append(prefix[-1] + value)
        closing_prefix_by_pair[pair] = prefix

    return incoming_by_target, closing_by_pair, closing_prefix_by_pair

# Count cycle motifs of the form A -> B -> C -> A within the time window
def compute_cycle_contribs(tx, delta=3600):
    # Keep only required columns to avoid unpacking errors
    tx = tx.select(["source", "target", "timestamp", "value_eth"])

    incoming_by_target, closing_by_pair, closing_prefix_by_pair = build_cycle_indexes(tx)

    # Store cycle counts and values for each node role: start, middle, and end
    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0, 0, 0.0])
    tx_second_edges = tx.sort("timestamp")

    # Treat each transaction B -> C as the second edge of a potential cycle
    for row in tx_second_edges.iter_rows(named=True):
        b = row["source"]
        c = row["target"]
        t2 = row["timestamp"]
        v2 = row["value_eth"]

        # Find previous incoming edges A -> B
        incoming_data = incoming_by_target.get(b)
        if incoming_data is None:
            continue

        in_times = incoming_data["times"]
        in_sources = incoming_data["sources"]
        in_values = incoming_data["values"]

        # Keep only A -> B transactions that happened before B -> C and within delta
        left_in = bisect_left(in_times, t2 - delta)
        right_in = bisect_left(in_times, t2)

        if left_in == right_in:
            continue

        for i in range(left_in, right_in):
            a = in_sources[i]
            t1 = in_times[i]
            v1 = in_values[i]

            # Require three distinct addresses in the cycle
            if a == b or b == c or a == c:
                continue

            # Look for closing edges C -> A
            closing_pair = (c, a)
            closing_data = closing_by_pair.get(closing_pair)
            if closing_data is None:
                continue

            close_times = closing_data["times"]
            close_prefix = closing_prefix_by_pair[closing_pair]

            # Closing edge must happen after B -> C and before the full delta window ends
            left_close = bisect_right(close_times, t2)
            right_close = bisect_right(close_times, t1 + delta)

            k = right_close - left_close
            if k <= 0:
                continue

            # Compute total ETH value for all matching cycles
            closing_value_sum = close_prefix[right_close] - close_prefix[left_close]
            motif_value_sum = k * (v1 + v2) + closing_value_sum

            # Add contribution for A as the start node
            contrib[a][0] += k
            contrib[a][1] += motif_value_sum

            # Add contribution for B as the middle node
            contrib[b][2] += k
            contrib[b][3] += motif_value_sum

            # Add contribution for C as the end node
            contrib[c][4] += k
            contrib[c][5] += motif_value_sum

    # Return an empty table if no cycles were detected
    if len(contrib) == 0:
        return empty_cycle_contribs()

    # Convert dictionary results into rows
    rows = []
    for address, values in contrib.items():
        rows.append((
            address,
            values[0],
            values[1],
            values[2],
            values[3],
            values[4],
            values[5]
        ))

    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "cycle_as_start_count_1h",
            "cycle_as_start_value_eth_1h",
            "cycle_as_middle_count_1h",
            "cycle_as_middle_value_eth_1h",
            "cycle_as_end_count_1h",
            "cycle_as_end_value_eth_1h"
        ],
        orient="row"
    )


# Compute address-level cycle motif features for the selected 1-hour window
cycle_features = compute_cycle_contribs(events, delta=3600)

# Add cycle motif features to the node feature table
nodes = (
    nodes
    .join(cycle_features, on="address", how="left")
    .fill_null(0)
)

nodes.shape
nodes.columns

['address',
 'repeat_same_direction_as_sender_count_1h',
 'repeat_same_direction_as_sender_value_eth_1h',
 'repeat_same_direction_as_receiver_count_1h',
 'repeat_same_direction_as_receiver_value_eth_1h',
 'motif_reciprocity_cnt',
 'motif_reciprocity_val_sum',
 'motif_chain_cnt',
 'motif_fan_in_unique_sources',
 'motif_fan_in_cnt',
 'motif_fan_out_unique_targets',
 'motif_fan_out_cnt',
 'cycle_as_start_count_1h',
 'cycle_as_start_value_eth_1h',
 'cycle_as_middle_count_1h',
 'cycle_as_middle_value_eth_1h',
 'cycle_as_end_count_1h',
 'cycle_as_end_value_eth_1h']

## 8. Exporting Engineered Features

In [9]:
output_file_path = "nodes_with_temporal_motifs_1h.csv"

print(f"Saving final features to: {output_file_path}...")
nodes.write_csv(output_file_path)

print("File successfully saved!")
print(f"Final dataset shape: {nodes.shape}")
print(f"Available columns: {nodes.columns}")

Saving final features to: nodes_with_temporal_motifs_1h.csv...
File successfully saved!
Final dataset shape: (1397958, 18)
Available columns: ['address', 'repeat_same_direction_as_sender_count_1h', 'repeat_same_direction_as_sender_value_eth_1h', 'repeat_same_direction_as_receiver_count_1h', 'repeat_same_direction_as_receiver_value_eth_1h', 'motif_reciprocity_cnt', 'motif_reciprocity_val_sum', 'motif_chain_cnt', 'motif_fan_in_unique_sources', 'motif_fan_in_cnt', 'motif_fan_out_unique_targets', 'motif_fan_out_cnt', 'cycle_as_start_count_1h', 'cycle_as_start_value_eth_1h', 'cycle_as_middle_count_1h', 'cycle_as_middle_value_eth_1h', 'cycle_as_end_count_1h', 'cycle_as_end_value_eth_1h']
